# Tutorial: Using the Agents Feature for Conversational AI

This notebook provides a comprehensive guide to using the `agents` module, which simplifies interaction with various virtual agent platforms. It covers two main agent types:

1.  **PolySynth Agents:** For interacting with a (potentially internal) PolySynth API.
2.  **Dialogflow Agents:** For managing conversations with Google Cloud Dialogflow CX agents.

You'll learn how to set up clients, create sessions/conversations, send messages, and receive responses from both types of agents.

## 1. Setup Environment and Dependencies

First, let's install the necessary Python libraries. These include the Google Cloud Dialogflow client, `requests` for HTTP calls (used by PolySynth), and `google-auth` for authentication. We'll also import `logging` for better visibility into the agent's operations and `os` for environment variables.

In [ ]:
# Install necessary libraries
!pip install google-cloud-dialogflow-v2beta1 requests google-auth

import logging
import os
import time
import uuid

# Configure logging to see detailed output
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

## 2. Mock `conidk.core.base` Module

The `agents.py` module depends on `conidk.core.base` for authentication and configuration. Since `conidk` is likely an internal library, we'll provide a minimal mock implementation for `base.Auth`, `base.Config`, and `base.Environments` to allow `agents.py` to run in this standalone notebook. We'll also use `sys.modules` to make this mock importable as `conidk.core.base`.

In a real deployment, you would ensure `conidk` is properly installed and configured in your environment.

In [ ]:
# Minimal mock for conidk.core.base
class Environments:
    """Mock for conidk.core.base.Environments."""
    def __init__(self, env: str):
        self.env = env

    def get(self, key: str, default: str) -> str:
        # Simplified: just return a fictional base URL based on env
        if key == "polysynth_endpoint":
            if self.env == "stg":
                # This is a fictional URL. Replace with your actual PolySynth staging endpoint.
                return "https://polysynth-staging.example.com/"
            # This is a fictional URL. Replace with your actual PolySynth production endpoint.
            return "https://polysynth-prod.example.com/"
        return default

class Config:
    """Mock for conidk.core.base.Config."""
    def __init__(self, environment=None):
        self._environment = environment if environment else Environments("prod")

    def set_polysynth_endpoint(self) -> str:
        return self._environment.get("polysynth_endpoint", "https://default-polysynth.example.com/")

class Auth:
    """Mock for conidk.core.base.Auth."""
    def __init__(self):
        pass # No specific implementation needed for this example

# Ensure conidk.core.base is available for import for agents.py
# We'll make a fake module here directly to satisfy the import.
import sys
from types import ModuleType

if 'conidk' not in sys.modules:
    sys.modules['conidk'] = ModuleType('conidk')
if 'conidk.core' not in sys.modules:
    sys.modules['conidk.core'] = ModuleType('conidk.core')
if 'conidk.core.base' not in sys.modules:
    sys.modules['conidk.core.base'] = ModuleType('conidk.core.base')

sys.modules['conidk.core.base'].Auth = Auth
sys.modules['conidk.core.base'].Config = Config
sys.modules['conidk.core.base'].Environments = Environments

logging.info("Mock conidk.core.base classes loaded.")

## 3. Embed `agents.py` Source Code

To make this notebook self-contained and runnable without extra file management, we'll embed the entire `agents.py` source code directly into this cell. This allows us to define and use the `PolySynth` and `Dialogflow` classes within the notebook environment.

In [ ]:
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

"""Manages conversations with virtual agents.

This module provides classes for interacting with different virtual agent
platforms, including PolySynth and Dialogflow.
"""

# The 'time' import is handled in the setup cell
from typing import Any, Dict, Optional
# The 'uuid' import is handled in the setup cell
# The 'logging' import is handled in the setup cell
from google.cloud import dialogflow_v2beta1
from google.cloud.dialogflow_v2beta1.services.conversations import ConversationsClient
from google.cloud.dialogflow_v2beta1.services.participants import ParticipantsClient
from google.cloud.dialogflow_v2beta1.services.conversation_profiles import (
    ConversationProfilesClient,
)
import requests
import google.auth
from google.auth.transport.requests import Request

# Import the mock base module we defined earlier
from conidk.core import base

_EXPECTED_RESOURCE_TYPES = ["type.googleapis.com/google.cloud.ces.v1.App"]


class PolySynth:
    """Interacts with the PolySynth API for conversational agents.

    Provides methods to create sessions, send messages, and handle
    authentication and long-running operations with the PolySynth API.
    """

    api_version = "v1beta"

    def __init__(
        self,
        project_id: str,
        location: str,
        env: str = "stg",
        auth: Optional[base.Auth] = None,
        config: Optional[base.Config] = None,
    ) -> None:
        """Initializes the PolySynth client.
        Args:
            project_id: The ID of the Google Cloud project.
            location: The Google Cloud location for the API endpoint.
            env: The target environment, such as "stg" or "prod".
            auth: An optional authentication handler.
            config: An optional configuration handler.
        """

        self.project_id = project_id
        self.location = location
        self.auth = auth or base.Auth()
        self.config = config or base.Config(environment=base.Environments(env))
        self.credentials_token: Optional[str] = None
        self.agent_id: Optional[str] = None
        self.current_session_id: Optional[str] = None
        self._set_credentials()

    @property
    def parent(self) -> str:
        """Constructs the parent resource string for API requests.

        Returns:
            The formatted parent string, e.g., 'projects/{id}/locations/{loc}'.
        """
        return f"projects/{self.project_id}/locations/{self.location}"

    @property
    def base_url(self) -> str:
        """Constructs the base URL for the PolySynth API.

        Returns:
            The base URL for the API endpoint.
        """
        return f"{self.config.set_polysynth_endpoint()}{self.api_version}/"

    def _set_credentials(self) -> Optional[str]:
        """Refreshes and sets the authentication credentials.
        """
        credentials, _ = google.auth.default()
        if not credentials.valid or credentials.expired:
            credentials.refresh(Request())

        if credentials.token:
            self.credentials_token = credentials.token
        else:
            logging.info("Could not retrieve access token.")

        return self.credentials_token

    def _make_request(
        self,
        method,
        url,
        headers=None,
        json=None,
        params=None,
        operation_timeout=300,
    ):
        """Makes an HTTP request and handles long-running operations.

        Args:
            method: The HTTP method (e.g., "GET", "POST").
            url: The URL endpoint for the request.
            headers: Optional dictionary of request headers.
            json: Optional JSON payload for the request body.
            params: Optional dictionary of URL parameters.
            operation_timeout: The timeout in seconds for polling long-running operations.

        Returns:
            The JSON response from the API as a dictionary, or None if the
            request fails.
        """
        access_token = self._set_credentials()

        if not access_token:
            logging.error("Failed to get access token.")
            return None

        if headers is None:
            headers = {}

        headers["Authorization"] = f"Bearer {access_token}"  # Add authorization header
        if json is not None and "Content-Type" not in headers:
            headers["Content-Type"] = "application/json"  # Ensure Content-Type for JSON
        try:
            response = requests.request(
                method, self.base_url + url, headers=headers,
                json=json, params=params, timeout=operation_timeout
            )
            response.raise_for_status()
            response_json = response.json()

            # Check for long-running operation
            if "name" in response_json and "done" in response_json:
                if not response_json["done"]:
                    return self._poll_operation(
                        response_json["name"], timeout=operation_timeout
                    )
                return response_json  # Operation already complete
            return response_json

        except requests.exceptions.RequestException as e:
            logging.error("Request failed: %s", e)
            if hasattr(e, 'response') and e.response is not None:
                logging.error("Response status: %s", e.response.status_code)
                logging.error("Response content: %s", e.response.text)
            else:
                logging.info("Response content: (no response object)")
            return None

    def _poll_operation(
        self,
        operation_name,
        timeout=300,
        initial_sleep=2,
        poll_interval=5,
    ):
        """Polls a long-running operation until it completes or times out.

        Args:
            operation_name: The name of the operation to poll.
            timeout: The maximum time in seconds to wait for completion.
            initial_sleep: The initial delay before the first poll.
            poll_interval: The interval in seconds between poll attempts.

        Returns:
            The response payload from the completed operation, or None on timeout.

        Raises:
            RuntimeError: If the operation fails.
        """
        start_time = time.time()
        if initial_sleep > 0:
            time.sleep(initial_sleep)

        while time.time() - start_time < timeout:
            # The operation_name might be a full resource path, or relative. 
            # If it's a full path, prepending base_url might be incorrect. 
            # For demonstration, we'll temporarily adjust base_url for polling requests.
            original_base_url = self.base_url
            # Assume operation_name is an absolute path or relative to the root of the API endpoint
            # For simplicity, if operation_name starts with 'projects/', we treat it as an absolute resource path.
            if operation_name.startswith('projects/'):
                # The base_url might contain 'v1beta/' at the end, which is not suitable for full resource names.
                # We need to construct the URL for operations correctly, usually just the host + operation_name.
                # This is a simplification and might need adjustment based on the actual PolySynth API spec.
                parts = original_base_url.split('/')
                # Attempt to get the root API endpoint (e.g., https://polysynth-staging.example.com/)
                root_api_endpoint = '/'.join(parts[:-2]) + '/' if len(parts) > 2 else original_base_url # Heuristic
                request_url = f"{root_api_endpoint}{operation_name}"
            else:
                # Assume operation_name is relative to the current base_url
                request_url = f"{original_base_url}{operation_name}"

            # Make a direct request to the operation URL, bypassing the default base_url concatenation
            # For this internal method, we directly use requests.request with the full URL.
            access_token = self._set_credentials()
            if not access_token:
                logging.error("Failed to get access token during polling.")
                return None

            headers = {"Authorization": f"Bearer {access_token}"}

            try:
                response = requests.request("GET", request_url, headers=headers, timeout=poll_interval)
                response.raise_for_status()
                result = response.json()
            except requests.exceptions.RequestException as e:
                logging.warning(
                    "Polling %s: Request failed: %s. Retrying after %ss.",
                    operation_name, e, poll_interval
                )
                if hasattr(e, 'response') and e.response is not None:
                    logging.debug("Polling response error: %s", e.response.text)
                time.sleep(poll_interval)
                continue

            if result and isinstance(result, dict):
                if "done" in result:
                    if result.get("done"):
                        logging.info(
                            "Operation %s completed (via Operation object).",
                            operation_name,
                        )
                        if "response" in result:
                            return result.get("response")
                        if "error" in result:
                            error_details = result.get("error")
                            logging.error(
                                "Operation %s failed: %s", operation_name, error_details
                            )
                            raise RuntimeError(
                                f"Operation {operation_name} failed: {error_details}"
                            )
                        logging.warning(
                            "Operation %s completed (via Op) without 'response' or 'error'.",
                            operation_name,
                        )
                        return {}  # Indicate success without payload
                    logging.debug(
                        "Operation %s pending (via Op). Sleeping for %ss.",
                        operation_name,
                        poll_interval,
                    )
                if result.get("@type") in _EXPECTED_RESOURCE_TYPES:
                    logging.info(
                        "Operation %s completed (detected final resource type).",
                        operation_name,
                    )
                    return result
                logging.warning(
                    "Polling %s: Received unexpected dictionary format. "
                    "Retrying after %ss. Response: %s",
                    operation_name,
                    poll_interval,
                    result,
                )
            elif result is None:
                logging.warning(
                    "Polling %s: No result/error from request. Retrying after %ss.",
                    operation_name,
                    poll_interval,
                )
            else:
                logging.error(
                    "Polling %s: Received completely unexpected response type (%s). "
                    "Retrying after %ss. Response: %s",
                    operation_name,
                    type(result),
                    poll_interval,
                    result,
                )

            time.sleep(poll_interval)

        logging.error(
            "Operation %s timed out after %s seconds.", operation_name, timeout
        )
        return None

    def create_session(self, agent_id: str, unique_id: Optional[str] = None) -> str:
        """Creates a new conversational session for a given agent.

        Args:
            agent_id: The identifier of the agent.
            unique_id: An optional unique identifier for the session. If not
                provided, a random UUID is generated.
        Returns:
            The fully qualified session ID string.
        """
        self.agent_id = agent_id
        if unique_id:
            session_id = unique_id
        else:
            session_id = str(uuid.uuid4())

        self.current_session_id = (
            f"{self.parent}/apps/{self.agent_id}/sessions/{session_id}"
        )
        logging.info("Created PolySynth session: %s", self.current_session_id)
        return self.current_session_id

    def send_message(self, text: str, session_id: Optional[str] = None):
        """Sends a text message to a conversational session.

        Args:
            text: The user's text input to send to the agent.
            session_id: The ID of the session to which the message should be
                sent. If None, the currently active session (`self.current_session_id`) is used.

        Returns:
            The agent's text response as a string, or the raw session output.
        """
        if session_id is None:
            session_id = self.current_session_id
            if session_id is None:
                raise ValueError("Session ID must be provided or set via create_session.")

        payload = {"config": {"session": session_id}}
        # Construct the input part - currently only text
        session_input: Dict[str, Any] = {}
        if text is not None:
            session_input["text"] = text

        payload["inputs"] = session_input
        url = f"{session_id}:runSession"
        
        logging.info("Sending message to PolySynth session: %s", session_id)
        logging.debug("Payload: %s", payload)
        session_output = self._make_request("POST", url, json=payload)
        logging.debug("PolySynth raw output: %s", session_output)

        text_response = ''
        try:
            if session_output and session_output.get("outputs"):
                for response in session_output["outputs"]:
                    if response.get("text"):
                        text_response += response["text"] + "\n"

            if len(text_response) != 0:
                answer = text_response.strip()
            else:
                answer = session_output

        except Exception as e:  # pylint: disable=broad-exception-caught
            answer = 'session ended'
            logging.error("An error occurred parsing PolySynth response: %s", e)

        return answer


class Dialogflow:
    """Manages conversations using the Dialogflow API.

    Provides methods to create and manage conversations with a Dialogflow
    agent, including creating conversation profiles, adding participants,
    and sending messages.
    """
    def __init__(
        self,
        project_id: str,
        location: str,
        conversation_profile: str,
        auth: Optional[base.Auth] = None,
        config: Optional[base.Config] = None,
    ):
        """Initializes the Dialogflow client.

        Args:
            project_id: The ID of the Google Cloud project.
            location: The Google Cloud location for the API endpoint.
            conversation_profile: The name of the conversation profile to use.
            auth: An optional authentication handler.
            config: An optional configuration handler.
        """
        self.project_id = project_id
        self.location = location
        self.auth = auth or base.Auth()
        self.config = config or base.Config()
        self.conversation_profile = conversation_profile
        self.parent = f"projects/{project_id}/locations/{location}"

        self.conversation_client = ConversationsClient()
        self.participant_client = ParticipantsClient()

        self.conversation_name: Optional[str] = None
        self.participant_name: Optional[str] = None

    def create_conversation(self) -> str:
        """Creates a new conversation with a Dialogflow agent.

        Returns:
            The resource name of the newly created conversation.
        """

        conversation = dialogflow_v2beta1.Conversation()
        conversation.conversation_profile = (
            f"{self.parent}/conversationProfiles/{self.conversation_profile}"
        )
        conversation.lifecycle_state = (
            dialogflow_v2beta1.Conversation.LifecycleState.IN_PROGRESS  # type: ignore[assignment]
        )
        response = self.conversation_client.create_conversation(
            parent=self.parent, conversation=conversation
        )
        self.conversation_name = response.name
        logging.info("Created Dialogflow conversation: %s", self.conversation_name)
        return self.conversation_name

    def create_participant(self):
        """Creates a new participant and adds them to the conversation.

        Returns:
            The resource name of the newly created participant.

        Raises:
            RuntimeError: If a conversation has not been created first.
        """
        if not self.conversation_name:
            raise RuntimeError(
                "Must call create_conversation() before create_participant()"
            )
        participant = dialogflow_v2beta1.Participant()
        participant.role = dialogflow_v2beta1.Participant.Role.END_USER
        new_participant = self.participant_client.create_participant(
            parent=self.conversation_name, participant=participant
        )
        self.participant_name = new_participant.name
        logging.info("Created participant: %s", self.participant_name)
        return self.participant_name

    def create_conversation_profile(
        self,
        virtual_agent_path: str,
        display_name: str,
    ):
        """Creates a new conversation profile for a virtual agent.

        Args:
            virtual_agent_path: The resource name of the virtual agent.
                                E.g., `projects/PROJECT_ID/locations/LOCATION/agents/AGENT_ID`
            display_name: The human-readable display name for the profile.

        Returns:
            The newly created ConversationProfile object.
        """
        client = ConversationProfilesClient()
        conversation_profile = dialogflow_v2beta1.ConversationProfile(
            display_name=display_name,
            automated_agent_config=dialogflow_v2beta1.AutomatedAgentConfig(
                agent=virtual_agent_path,
            ),
        )
        response = client.create_conversation_profile(
            parent=self.parent,
            conversation_profile=conversation_profile,
        )
        logging.info("Created conversation profile: %s", response.name)
        # Update the instance's conversation_profile for immediate use if it was created
        self.conversation_profile = response.name.split('/')[-1] # Extract just the ID
        return response

    def send_message(self, text: str, language_code: str = "en-US") -> str:
        """Sends a text message to the conversation and gets a reply.

        Args:
            text: The user's text input to send to the agent.
            language_code: The language of the input text (e.g., "en-US").

        Returns:
            The agent's text reply as a string.

        Raises:
            RuntimeError: If a participant has not been created first.
        """
        if not self.participant_name:
            raise RuntimeError("Must call create_participant() before send_message()")
        text_input = dialogflow_v2beta1.TextInput(
            text=text, language_code=language_code
        )
        request = dialogflow_v2beta1.AnalyzeContentRequest(
            participant=self.participant_name,
            text_input=text_input,
        )

        logging.info("Sending message to Dialogflow participant: %s", self.participant_name)
        logging.debug("Text input: %s", text)
        response = self.participant_client.analyze_content(request=request)
        logging.debug("Dialogflow raw response: %s", response)

        return str(response.reply_text)

    def complete_conversation(self) -> None:
        """Marks the current conversation as complete.

        Raises:
            RuntimeError: If a conversation has not been created first.
        """
        if not self.conversation_name:
            raise RuntimeError(
                "Must call create_conversation() before complete_conversation()"
            )
        logging.info("Completing Dialogflow conversation: %s", self.conversation_name)
        self.conversation_client.complete_conversation(
            name=self.conversation_name
        )

## 4. Google Cloud Project Configuration

Before proceeding, you need to configure your Google Cloud project details and ensure you're authenticated. This tutorial assumes you have a Google Cloud project set up and have authenticated your environment (e.g., via `gcloud auth application-default login` for local environments or automatically in Colab notebooks with proper permissions).

**Replace the placeholders below** with your actual Google Cloud Project ID and desired location.

In [ ]:
# --- IMPORTANT: REPLACE WITH YOUR VALUES ---
PROJECT_ID = "YOUR_PROJECT_ID"  # e.g., "my-gcp-project-12345"
LOCATION = "YOUR_LOCATION"      # e.g., "us-central1"

# Set environment variables for Google Cloud client libraries
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

print(f"Project ID: {PROJECT_ID}")
print(f"Location: {LOCATION}")

# Verify default credentials (optional, but good practice)
try:
    credentials, project = google.auth.default()
    print(f"Authenticated with project: {project}")
except Exception as e:
    print(f"Authentication failed: {e}")
    print("Please ensure you're authenticated, e.g., via 'gcloud auth application-default login'.")

## 5. Part 1: Interacting with PolySynth Agents

The `PolySynth` class facilitates communication with the PolySynth conversational AI platform. This platform uses `agent_id`s to identify specific virtual agents and manages conversations through sessions.

### 5.1. Initialize the PolySynth Client

We'll start by initializing the `PolySynth` client with your project ID and location. You can also specify the environment (e.g., "stg" for staging or "prod" for production).

In [ ]:
# --- IMPORTANT: REPLACE WITH YOUR POLYSYNTH AGENT ID ---
# This ID should uniquely identify your PolySynth agent.
POLYSYNTH_AGENT_ID = "YOUR_POLYSYNTH_AGENT_ID" # e.g., "my-polysynth-agent-123"

print(f"PolySynth Agent ID: {POLYSYNTH_AGENT_ID}")

try:
    polysynth_client = PolySynth(
        project_id=PROJECT_ID,
        location=LOCATION,
        env="stg" # Or "prod" depending on your agent's deployment
    )
    print("PolySynth client initialized successfully.")
except Exception as e:
    print(f"Error initializing PolySynth client: {e}")
    polysynth_client = None

### 5.2. Create a Conversational Session

Each conversation with a PolySynth agent takes place within a session. The `create_session` method generates a unique session ID for your interaction. You need to provide the `agent_id` you want to converse with.

In [ ]:
session_id = None
if polysynth_client:
    try:
        session_id = polysynth_client.create_session(agent_id=POLYSYNTH_AGENT_ID)
        print(f"Created PolySynth session: {session_id}")
    except Exception as e:
        print(f"Error creating PolySynth session: {e}")
else:
    print("PolySynth client not initialized. Please check the previous steps.")

### 5.3. Send Messages and Receive Responses

Now you can send text messages to your PolySynth agent using the `send_message` method. The agent's text response will be returned. Note that `PolySynth` internally handles long-running operations (LROs) if the API response indicates one.

In [ ]:
if session_id and polysynth_client:
    print("--- Starting PolySynth Conversation ---")
    messages = [
        "Hello there!",
        "What can you do for me?",
        "Tell me a joke.",
        "Thank you, goodbye."
    ]

    for msg in messages:
        print(f"\nUser: {msg}")
        try:
            response = polysynth_client.send_message(text=msg)
            print(f"Agent: {response}")
        except Exception as e:
            print(f"Error sending message: {e}")
            break
    print("--- PolySynth Conversation Ended ---")
else:
    print("PolySynth session not created or client not initialized. Please check the previous steps.")

### 5.4. Understanding PolySynth's Long-Running Operations (LROs)

The `PolySynth` class is designed to handle API requests that might be long-running operations. Internally, methods like `_make_request` and `_poll_operation` manage the asynchronous nature of some PolySynth API calls. When an initial API call returns an operation that isn't immediately `done`, the client polls the operation status until it completes, ensuring you receive the final result or error. This mechanism transparently manages the waiting period for you.

## 6. Part 2: Interacting with Dialogflow Agents

The `Dialogflow` class provides an interface to Google Cloud Dialogflow CX, allowing you to manage full conversations. This involves creating a conversation, adding participants, and exchanging messages.

### 6.1. Prerequisites: Dialogflow Agent and Conversation Profile

To use the `Dialogflow` client, you need:

1.  **A Dialogflow CX Agent:** This is your conversational agent configured in the Google Cloud Console. You'll need its full resource path (e.g., `projects/PROJECT_ID/locations/LOCATION/agents/AGENT_ID`).
2.  **A Conversation Profile:** This profile links your conversation to the Dialogflow CX agent and defines how the conversation should behave. You can either create one programmatically using `create_conversation_profile` or reference an existing one.

**Replace the placeholders below** with your actual Dialogflow CX agent's path and a display name for your conversation profile. Make sure the `LOCATION` matches the location of your Dialogflow CX agent.

In [ ]:
# --- IMPORTANT: REPLACE WITH YOUR DIALOGFLOW CX AGENT PATH AND A PROFILE DISPLAY NAME ---
# Example for DIALOGFLOW_VIRTUAL_AGENT_PATH: "projects/my-gcp-project-12345/locations/us-central1/agents/my-cx-agent-abc"
# Ensure the Project ID and Location match your actual Dialogflow CX agent.
DIALOGFLOW_VIRTUAL_AGENT_PATH = f"projects/{PROJECT_ID}/locations/{LOCATION}/agents/YOUR_DIALOGFLOW_AGENT_ID" 
DIALOGFLOW_CONVERSATION_PROFILE_DISPLAY_NAME = "MyDemoConversationProfile" # A unique display name for the profile

# This will be updated after creating or identifying the profile.
DIALOGFLOW_CONVERSATION_PROFILE_ID = ""

print(f"Dialogflow Virtual Agent Path: {DIALOGFLOW_VIRTUAL_AGENT_PATH}")
print(f"Conversation Profile Display Name: {DIALOGFLOW_CONVERSATION_PROFILE_DISPLAY_NAME}")

### 6.2. Create a Conversation Profile (If needed)

If you don't have an existing conversation profile linked to your Dialogflow CX agent, you can create one using the `create_conversation_profile` method. This step only needs to be performed once per unique agent/profile configuration.

**Note:** If you already have a conversation profile, you can skip this cell and manually set `DIALOGFLOW_CONVERSATION_PROFILE_ID` to its ID. The ID is typically the last segment of the conversation profile's resource name (e.g., `my-profile-id` from `projects/.../conversationProfiles/my-profile-id`).

We'll attempt to create one, and then extract its ID for use in the `Dialogflow` client. If a profile with the same display name already exists, the creation might fail, in which case you might need to manually retrieve its ID.

In [ ]:
try:
    # A temporary Dialogflow client instance is needed to call create_conversation_profile.
    # The 'conversation_profile' arg for its init is not critical here as we are creating it.
    temp_df_client = Dialogflow(PROJECT_ID, LOCATION, "temp-profile-placeholder")

    print(f"Attempting to create conversation profile: {DIALOGFLOW_CONVERSATION_PROFILE_DISPLAY_NAME}")
    profile_response = temp_df_client.create_conversation_profile(
        virtual_agent_path=DIALOGFLOW_VIRTUAL_AGENT_PATH,
        display_name=DIALOGFLOW_CONVERSATION_PROFILE_DISPLAY_NAME
    )
    DIALOGFLOW_CONVERSATION_PROFILE_ID = profile_response.name.split('/')[-1]
    print(f"Successfully created conversation profile with ID: {DIALOGFLOW_CONVERSATION_PROFILE_ID}")

except Exception as e:
    print(f"Could not create conversation profile. This might be because it already exists or due to permission issues: {e}")
    if "already exists" in str(e).lower() and DIALOGFLOW_CONVERSATION_PROFILE_DISPLAY_NAME:
        # If it exists, we assume the user might want to use that existing one.
        # For Dialogflow CX, the 'display_name' is often also the ID for simple cases.
        # In a robust application, you would list profiles to find the exact ID.
        DIALOGFLOW_CONVERSATION_PROFILE_ID = DIALOGFLOW_CONVERSATION_PROFILE_DISPLAY_NAME 
        print(f"Assuming existing profile ID: {DIALOGFLOW_CONVERSATION_PROFILE_ID}" \
              " (Verify this ID is correct for your existing profile). \n" \
              "If it's not, you'll need to manually set `DIALOGFLOW_CONVERSATION_PROFILE_ID`.")
    else:
        DIALOGFLOW_CONVERSATION_PROFILE_ID = "" # Clear if creation failed for other reasons

if not DIALOGFLOW_CONVERSATION_PROFILE_ID:
    print("WARNING: DIALOGFLOW_CONVERSATION_PROFILE_ID is empty. Subsequent Dialogflow calls may fail.")

### 6.3. Initialize the Dialogflow Client

With the Conversation Profile ID (or its display name if it's used as ID), we can now initialize the `Dialogflow` client.

In [ ]:
dialogflow_client = None
if DIALOGFLOW_CONVERSATION_PROFILE_ID:
    try:
        dialogflow_client = Dialogflow(
            project_id=PROJECT_ID,
            location=LOCATION,
            conversation_profile=DIALOGFLOW_CONVERSATION_PROFILE_ID
        )
        print("Dialogflow client initialized successfully.")
    except Exception as e:
        print(f"Error initializing Dialogflow client: {e}")
else:
    print("DIALOGFLOW_CONVERSATION_PROFILE_ID is not set. Cannot initialize Dialogflow client.")

### 6.4. Start a New Conversation

A conversation is the overarching container for an interaction. `create_conversation` starts a new one with the Dialogflow agent associated with your chosen `conversation_profile`.

In [ ]:
conversation_name = None
if dialogflow_client:
    try:
        conversation_name = dialogflow_client.create_conversation()
        print(f"Created Dialogflow conversation: {conversation_name}")
    except Exception as e:
        print(f"Error creating Dialogflow conversation: {e}")
else:
    print("Dialogflow client not initialized. Skipping conversation creation.")

### 6.5. Add a Participant to the Conversation

Conversations require participants (typically an end-user and the virtual agent). `create_participant` adds the end-user, allowing messages to be sent.

In [ ]:
participant_name = None
if conversation_name and dialogflow_client:
    try:
        participant_name = dialogflow_client.create_participant()
        print(f"Created Dialogflow participant: {participant_name}")
    except Exception as e:
        print(f"Error creating Dialogflow participant: {e}")
else:
    print("Dialogflow conversation or client not initialized. Skipping participant creation.")

### 6.6. Exchange Messages

Use `send_message` to interact with the Dialogflow agent, sending user input and receiving its replies. The `reply_text` field contains the agent's response.

In [ ]:
if participant_name and dialogflow_client:
    print("--- Starting Dialogflow Conversation ---")
    messages = [
        "Hi, I need some help.",
        "What are your operating hours?",
        "Can I speak to a human?"
    ]

    for msg in messages:
        print(f"\nUser: {msg}")
        try:
            reply_text = dialogflow_client.send_message(text=msg)
            print(f"Agent: {reply_text}")
        except Exception as e:
            print(f"Error sending message: {e}")
            break
    print("--- Dialogflow Conversation Ended ---")
else:
    print("Dialogflow participant not created or client not initialized. Please check the previous steps.")

### 6.7. Complete the Conversation

Once the interaction is finished, it's good practice to mark the conversation as complete using `complete_conversation` to release resources and manage conversation state.

In [ ]:
if conversation_name and dialogflow_client:
    try:
        dialogflow_client.complete_conversation()
        print(f"Conversation {conversation_name} marked as complete.")
    except Exception as e:
        print(f"Error completing conversation: {e}")
else:
    print("Dialogflow conversation not initialized or client missing. Skipping conversation completion.")

## Conclusion

You've now walked through how to use the `agents` module to interact with both PolySynth and Dialogflow conversational agents.

Key takeaways:

*   The module provides a unified way to manage conversations across different platforms.
*   **PolySynth** interactions involve initializing a client, creating sessions, and sending messages, with built-in handling for long-running operations.
*   **Dialogflow** interactions require setting up a conversation profile, creating a conversation, adding a participant, and then exchanging messages, finally completing the conversation.
*   Proper Google Cloud authentication and configuration are essential for Dialogflow and PolySynth (if using Google Cloud authentication).

Remember to replace all `YOUR_...` placeholders with your specific project details, agent IDs, and conversation profile names for the code to run successfully against live agents.